# Notebook 02 — Transform preview (cleaning + risk)

Objectif : vérifier ce que la phase Transform produit (`processed`) et contrôler le score/label de risque.

Fichiers :
- input : `data/raw/patient_vitals_mimic_demo_raw.csv`
- output : `data/processed/patient_vitals_processed.csv`


In [ ]:
import sys
import subprocess
from pathlib import Path

print("Python executable:", sys.executable)

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "notebooks").exists() else cwd.parent
raw_csv = PROJECT_ROOT / "data" / "raw" / "patient_vitals_mimic_demo_raw.csv"
processed_csv = PROJECT_ROOT / "data" / "processed" / "patient_vitals_processed.csv"

raw_csv.exists(), processed_csv.exists()

## 1) Charger raw vs processed

In [ ]:
try:
    raw_csv
    processed_csv
except NameError:
    from pathlib import Path
    import sys
    import subprocess
    try:
        import pandas as pd
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
        import pandas as pd
    cwd = Path.cwd()
    PROJECT_ROOT = cwd if (cwd / "notebooks").exists() else cwd.parent
    raw_csv = PROJECT_ROOT / "data" / "raw" / "patient_vitals_mimic_demo_raw.csv"
    processed_csv = PROJECT_ROOT / "data" / "processed" / "patient_vitals_processed.csv"

df_raw = pd.read_csv(raw_csv)
df_proc = pd.read_csv(processed_csv)

df_raw.shape, df_proc.shape

In [ ]:
df_proc.head(10)

## 2) Contrôles qualité (missing / doublons / types)

In [ ]:
missing_rate = df_proc.isna().mean().sort_values(ascending=False)
missing_rate.head(20)

In [ ]:
dup_count = int(df_proc.duplicated(subset=["patient_id", "recorded_at"]).sum())
dup_count

In [ ]:
df_proc.dtypes

## 3) Risk score / Risk level

On vérifie la distribution des classes et quelques exemples concrets.

In [ ]:
df_proc["risk_level"].value_counts()

In [ ]:
df_proc.groupby("risk_level")["risk_score"].describe()

In [ ]:
df_proc.loc[df_proc["risk_level"] == "Critical"].head(10)

## 4) Mini “dashboard” en tableau (top risques)

Pour Power BI plus tard : ce type de vue est très utile.

In [ ]:
cols = [
    "patient_id",
    "recorded_at",
    "age",
    "temperature_c",
    "systolic_bp",
    "diastolic_bp",
    "spo2",
    "heart_rate",
    "glucose_mg_dl",
    "pain_score",
    "risk_score",
    "risk_level",
]
df_proc.sort_values(["risk_score", "spo2"], ascending=[False, True])[cols].head(20)